In [ ]:
## Packages Import
import numpy             as np
import matplotlib.pyplot as plt

In [ ]:
class LiftingLineSolver:
    """
    Prandtl lifting-Line method with matrix formulation
    """
    #-------------------#
    #   Initialization  #
    #-------------------#
    def __init__(self, span, chord, alpha, a0, alpha_L0,
                U_inf, N,
                quad_type, bc_type, ghost, bc_val=0.0, pad_val=0.0):
        # Wing geometry
        self.half_model = False
        self.span = span           # [m] wing span
        self.semi_span = span/2    # [m] wing semi-span
        self.chord = chord         # [m] sectional chord
        self.alpha = alpha         # [deg] geometric angle of attack
        # Airfoil characteristics
        self.a0 = a0                # [1/deg] lift curve slope
        self.alpha_L0 = alpha_L0    # [deg] angle of attack @ zero lift
        # Flow properties
        self.U = U_inf              # [m/s] undisturbed flow velocity

        # Discretization
        self.N = N      # number of cells - must be even!
        self.z = None      # cells centroids
        self.h = None      # cells size

        # Numerical scheme
        self.quadrature_type = quad_type    # quadrature rule type
        self.bc_type         = bc_type      # boundary condition type
        self.bc_val          = bc_val       # boundary condition value
        self.ghost           = ghost        # ghost-cell technique
        self.pad_val         = pad_val      # padding value
        self.weight = None       # quadrature weights
        self.D      = None       # differential operator
        self.M      = None       # downwash kernel

        # Linear system
        self.A = None       # coefficient matrix
        self.b = None       # right hand side
        self.x = None       # unknown


    #-----------#
    #   Methods #
    #-----------#
    def _build_mesh(self):
        s = self.semi_span

        if self.half_model:
            z_min, z_max = 0.0, s
        else:
            z_min, z_max = -s, s

        zN = np.linspace(z_min, z_max, (self.N+1))   # cells nodes
        zC = (zN[1:] + zN[:-1]) / 2                    # cells centroids

        self.z = zC
        self.h = np.abs(zN[1] - zN[0])

    def _quadrature_weights(self):
        """
        Return the combined weights for all nodes according to the requested
        quadrature rule
        """
        h = self.h
        n = self.N

        if self.quadrature_type == 'simpson':
            assert n % 2 == 0, "Simpson's rule requires an even number of subintervals"
            w = np.zeros(n)
            w[0] = w[-1] = h/3
            w[1:-1:2] = 4*h/3
            w[2:-1:2] = 2*h/3
        else:
            pass    # TODO: implement else case

        self.weight = w
    
    def _central_finite_diff(self):
        """
        Second-order accurate central finite difference method over the internal nodes
        of the domain, one-sided difference over boundary elements; extension of central
        difference over the entire domain in case of ghost cell technique
        """
        h = self.h
        n = self.N

        # Interior nodes + boundary nodes (ghost)
        D = -1/(2*h) * np.diag(np.ones(n-1), -1) + \
            np.zeros((n, n)) + \
            1/(2*h) * np.diag(np.ones(n-1), 1)
        
        if (not self.ghost):
            # Overwrite boundary nodes
            # One-sided finite difference - 2nd order accurate
            D[0,  :3]  = [-3,  4, -1]/(2*h)     # root node (forward finite diff)
            D[-1, -3:] = [ 1, -4,  3]/(2*h)     # tip node (backward finite diff)
            
        self.D = D
    
    def _downwash_kernel(self):
        """
        Build the integral-differential operator relating downwash velocity at point P
        to the circulation over the lifting line
                        M -> v(zP) = 1/(4π) * M @ gamma
        """
        h = self.h

        if self.ghost:
            # Add extra nodes outside boundaries
            zG = [(self.z[0] - h), (self.z[-1] + h)]    # ghost nodes
            z = self._sym_pad(self.z, zG)
            n = len(z)
            # Update number of cells
            self.N = n
        else:
            z = self.z
            n  = self.N

        # Integral matrix
        self._quadrature_weights()
        H = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                if i != j:
                    H[i, j] = self.weight[j] / (z[j] - z[i])

        # Differential matrix
        self._central_finite_diff()
        
        # Full intregral-differential matrix
        self.M = np.dot(H, self.D)          # (N, N)
    
    def _sym_pad(self, arr, val=0.0):
        """
        Add symmetric padding for extra-domain nodes (ghost nodes)
        """
        if val:
            # Take values passed to function if present
            v = val
        else:
            v = self.pad_val
        
        if type(v) in [int, float]:
            # Cast assigned value to list type for generality
            v = [0]

        arr_pad = np.append(v[0], arr)
        arr_pad = np.append(arr_pad, v[-1])
        return arr_pad
    
    def _linear_sys(self):
        """
        """
        n  = self.N

        # Linear system matrix
        a1 = 1/(4*np.pi)
        a2 = 2/(np.rad2deg(self.a0)*self.chord)
        if self.ghost:
            a2 = self._sym_pad(a2)
        A = a1 * self.M - a2 * np.eye((n))
        # Right hand side
        b = self.U*np.deg2rad(self.alpha_L0 - self.alpha) * np.ones(n)
        
        self.A = A
        self.b = b
    
    def _set_bc(self):
        """
        Impose boundary conditions of given type at prescribed index, corresponding to
        boundary cells
        """
        idx = [0, -1]   # first and last element indeces

        if self.bc_type == 'dirichlet':
            # Impose identity
            self.A[idx, :]   = 0.0
            self.A[idx, idx] = 1.0
            self.b[idx] = self.bc_val
        else:
            pass    # TODO: implement else case

In [ ]:
## Parameters Definition
# Wing of ractangular planform
b = 10          # [m] wing span
c0 = 1          # [m] chord @ root
a0 = 0.1095     # [1/deg] lift curve slope
a = 6           # [deg] geometric angle of attack
a_L0 = -2       # [deg] angle of attack @ zero lift
U = 5           # [m/s] incident flow velocity

## TEST
## Integral-differential Operator vs Analytical Solution

llm = LiftingLineSolver(span=b,
                        chord=c0,
                        alpha=a,
                        a0=a0,
                        alpha_L0=a_L0,
                        U_inf=U,
                        N=None,
                        quad_type='simpson',
                        bc_type='dirichlet',
                        ghost=False)

# Elliptical circulation distribution
circ_elliptic = lambda y, s: np.sqrt(1 - (y/s)**2)
a0_rad = np.rad2deg(a0)
g0 = (2*b*a0_rad*c0*U / (4*b + a0_rad*c0)) * np.deg2rad(a - a_L0)       # circulation @ origin
v_ref = -g0 / (2*b)             # constant downwash velocity

fig = plt.figure(figsize=(10, 4))
# Figure with 2 subplots on the same row
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

# Wing discretization in spanwise direction
N = [10**p for p in range(1, 4)]          # number f cells
for n in N:
    setattr(llm, 'N', n)
    # Update wing discretization
    llm._build_mesh()
    
    # Evaluate reference solution
    gamma = g0*circ_elliptic(llm.z, llm.semi_span)

    # Compute numerical solution
    llm._downwash_kernel()
    v = 1/(4*np.pi) * np.dot(llm.M, gamma)      # induced downwash velocity

    # Plot downwash distribution along span
    ax2.plot(llm.z, v,
            linewidth=2,
            label=f"# cells={n:.1e}")
    
# Plot circulation (numerical) distribution along span
ax1.plot(llm.z, gamma,
        c='r',
        linewidth=2,
        label="elliptical distr")
ax1.set_xlabel(r"$Z\ [m]$")
ax1.set_ylabel(r"$\Gamma\ [m^2/s]$")
ax1.legend()
ax1.set_title("Circulation Distribution")
ax1.grid(True)
# Plot circulation (analytical) distribution along span
ax2.plot(llm.z, v_ref * np.ones(n),
        c='k',
        linewidth=2,
        linestyle='dashed',
        label="analytic sol")
ax2.set_xlabel(r"$Z\ [m]$")
ax2.set_ylabel(r"$v\ [m/s]$")
ax2.legend()
ax2.set_title("Velocity Downwash Distribution")
ax2.grid(True)

fig.tight_layout()
plt.show()

The order of convergence of the method with respect to the mesh size $\Delta z$ is defined as the exponent $p$ such that $e_{\Delta z} \propto \Delta z^p$

In [ ]:
def chord_fn(z, c0, s, shape="rectangular"):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape 
    """
    match shape:
        case "rectangular":
            c = c0 * np.ones(z.shape)
        case "elliptical":
            c = c0 * np.sqrt(1 - (z/s)**2)
    return c

## TEST
## Lifting-Line Method vs Analytical Solution

# Enable ghost cell technique
setattr(llm, 'ghost', True)

# Elliptical chord distribution
c_shape = "elliptical"  # wing planform shape

fig = plt.figure(figsize=(10, 4))
# Figure with 2 subplots on the same row
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

err = []
dz = []
for n in N:
    setattr(llm, 'N', n)
    # Update wing discretization
    llm._build_mesh()

    # Prescribe chord variation along span
    c = chord_fn(llm.z, c0, llm.semi_span, c_shape)
    setattr(llm, 'chord', c)

    llm._downwash_kernel()
    llm._linear_sys()
    llm._set_bc()

    # Solve linear system
    x = np.linalg.solve(llm.A, llm.b)

    # Plot circulation (numerical) distribution along span
    ax1.plot(llm.z, x[1:-1],
            linewidth=2,
            label=f"# cells={n:.1e}")

    # Estimate error with respect to reference solution
    ref = g0 * circ_elliptic(llm.z, llm.semi_span)
    eL2 = np.sqrt(np.sum((x[1:-1] - ref)**2) * llm.h)
    err.append(eL2)
    dz.append(llm.h)

# Plot circulation (analytical) distribution along span
ax1.plot(llm.z, ref,
        c='k',
        linewidth=2,
        linestyle='dashed',
        label="analytic sol")
ax1.set_xlabel(r"$Z\ [m]$")
ax1.set_ylabel(r"$\Gamma\ [m^2/s]$")
ax1.legend()
ax1.set_title(f"Circulation Distribution over {c_shape} Wing")
ax1.grid(True)

# Plot error
ax2.loglog(dz, err,
        c = 'gray',
        marker='o',
        linestyle='dashed',
        label="err L2")
ax2.loglog(dz, [np.sqrt(ddz) for ddz in dz],
           c='k',
           label="order 1/2")
ax2.loglog(dz, dz,
           c='b',
           label="order 1")
ax2.loglog(dz, [ddz**2 for ddz in dz],
           c='r',
           label="order 2")
ax2.set_xlabel(r"$\Delta Z \ [m]$")
ax2.set_ylabel(r"$\|\~\Gamma - \Gamma\|\ [m^2/s]$")
ax2.legend()
ax2.set_title(f"Numerical Convergence")
ax2.grid(True)

fig.tight_layout()
plt.show()